In [20]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent 
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from pydantic import BaseModel, Field
from typing import Literal

In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenAI(model="gpt-4o-mini")

In [7]:
prompt = """
Understand the coding request:

For any request, identify the following :

- task type
- objective
- files involved 

Return the response in the following JSON format:
{
    "task_type": "type of task",
    "objective": "objective of the task",
    "files_involved": "list of files involved"
}

"""

In [8]:
request = "Identify the bug in the auth.py file where the exchange token is not getting generated"

In [9]:
response = model.invoke(prompt + request)
print(response.content)

{
    "task_type": "bug identification",
    "objective": "identify the bug in the auth.py file where the exchange token is not getting generated",
    "files_involved": ["auth.py"]
}


In [12]:
class CodingRequest(BaseModel):
    """Structured representation of the coding request"""

    task_type: Literal[
        "implement",
        "debug",
        "review",
        "explain"
    ] = Field(description="Type of task to be performed")

    objective: str = Field(description="A concise description of what needs to be achieved")

    need_code_changes: bool = Field(
        description="Whether fulfilling this task required modifying the source code"
    )

    target_files: list[str] = Field(
        description="List of files that are relevant to the task, e.g. ['auth.py', 'models.py']"
    )



In [13]:
structured_output_model = model.with_structured_output(CodingRequest)

In [14]:
result = structured_output_model.invoke("""
    Fix the login bug in auth.py. Expired sessions currently produce HTTP 500.

""")

In [16]:
print(type(result))

<class '__main__.CodingRequest'>


In [19]:
CodingRequest.model_json_schema()

{'description': 'Structured representation of the coding request',
 'properties': {'task_type': {'description': 'Type of task to be performed',
   'enum': ['implement', 'debug', 'review', 'explain'],
   'title': 'Task Type',
   'type': 'string'},
  'objective': {'description': 'A concise description of what needs to be achieved',
   'title': 'Objective',
   'type': 'string'},
  'need_code_changes': {'description': 'Whether fulfilling this task required modifying the source code',
   'title': 'Need Code Changes',
   'type': 'boolean'},
  'target_files': {'description': "List of files that are relevant to the task, e.g. ['auth.py', 'models.py']",
   'items': {'type': 'string'},
   'title': 'Target Files',
   'type': 'array'}},
 'required': ['task_type', 'objective', 'need_code_changes', 'target_files'],
 'title': 'CodingRequest',
 'type': 'object'}

In [37]:
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(CodingRequest),
    system_prompt=prompt
)

In [38]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Identify the bug in the auth.py file where the exchange token is not getting generated"}
    ]
})
print(type(result["structured_response"]))
print(result["structured_response"])


<class '__main__.CodingRequest'>
task_type='debug' objective='Identify the bug in the auth.py file where the exchange token is not getting generated' need_code_changes=True target_files=['auth.py']


In [39]:
print(result)

{'messages': [HumanMessage(content='Identify the bug in the auth.py file where the exchange token is not getting generated', additional_kwargs={}, response_metadata={}, id='498e32f6-5f70-430c-b639-e9c86cc8d85e'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 207, 'total_tokens': 271, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_de64dd5ae6', 'id': 'chatcmpl-ELxyNyhlq7HsGtTjmKc0yIfxYK1bZ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a082fe-c8d6-7b92-be2c-a0913ad39135-0', tool_calls=[{'name': 'CodingRequest', 'args': {'task_type': 'debug', 'objective': 'Identify the bug in the

In [40]:
for message in result["messages"]:
    print(message.pretty_print())
    print("===================\n\n")


================================ Human Message =================================

Identify the bug in the auth.py file where the exchange token is not getting generated
None


================================== Ai Message ==================================
Tool Calls:
  CodingRequest (call_GHtRIdgtuJNfIjc9N8msbp6k)
 Call ID: call_GHtRIdgtuJNfIjc9N8msbp6k
  Args:
    task_type: debug
    objective: Identify the bug in the auth.py file where the exchange token is not getting generated
    need_code_changes: True
    target_files: ['auth.py']
None


================================= Tool Message =================================
Name: CodingRequest

Returning structured response: task_type='debug' objective='Identify the bug in the auth.py file where the exchange token is not getting generated' need_code_changes=True target_files=['auth.py']
None


